# Transformer Decoder

The transformer decoder for lm is a stack of causal self-attention and feed-forward layers.
The positional encoding is a learnable parameter and it is added to the input of the decoder.

In [1]:
class FeedForward(torch.nn.Module):
    def __init__(self, d_model=512, d_ff=1024, dropout=0.1, **kwargs):
        super().__init__()
        self.ff = torch.nn.Sequential(
            torch.nn.LayerNorm(d_model),
            torch.nn.Linear(d_model, d_ff),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(d_ff, d_model),
        )        
    def forward(self, x):
        return self.ff(x)

class CausalSelfAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, seq_len=1024,**kwargs):
        super().__init__()
        print('Causal Self Attention, d_model:', d_model, 'n_heads:', n_heads, 'd_head:', d_head, 'seq_len:', seq_len)
        self.seq_len = seq_len
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)        
        self.register_buffer("mask", torch.tril(torch.ones(self.seq_len, self.seq_len))[None, None, ...] == 0)
            
    def forward(self, x):
        x = self.norm(x)
        b, n, d = x.shape
        q = self.q_linear(x).reshape(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x).reshape(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x).reshape(b, -1, self.n_heads, self.d_head) 
        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale        
        scores = scores.masked_fill(self.mask[:,:,:n,:n], float('-inf'))
        att = scores.softmax(dim=-1)
        att = self.dropout(att)
        out = torch.einsum('bhij,bjhd->bihd', att, v)
        out = self.dropout(out).reshape(b, -1, self.n_heads*self.d_head)
        out = self.out(out)
        return out

class Decoder(torch.nn.Module):
    def __init__(self, nb_layers=6, **kwargs):
        super().__init__()        
        seq_len = kwargs['seq_len']     
        self.pos = torch.nn.Parameter(torch.randn(1, seq_len, kwargs['d_model']))
        self.att = torch.nn.ModuleList([CausalSelfAttention(**kwargs) for _ in range(nb_layers)])
        self.ff = torch.nn.ModuleList([FeedForward(**kwargs) for _ in range(nb_layers)])
        
    def forward(self, x):
        b, n, d = x.shape
        x = x + self.pos[:, :n, :]
        for att, ff in zip(self.att, self.ff):
            x = x + att(x)
            x = x + ff(x)            
        return x

class Transformer(torch.nn.Module):
    def __init__(self, vocab_size=20, **kwargs):
        super().__init__()
        self.vocab_size = vocab_size    
        self.seq_len = kwargs['seq_len']    
        self.emb = torch.nn.Embedding(vocab_size, kwargs['d_model'])        
        self.dec = Decoder(**kwargs)
        self.out = torch.nn.Linear(kwargs['d_model'], vocab_size)
        
    def decoder(self, x):
        x = self.emb(x)
        x = self.dec(x)
        return self.out(x)
    
    def forward(self, x):
        return self.decoder(x)
                
    def loss(self, y):         
        logits = self(y[:,:-1]).reshape(-1, self.vocab_size)
        target = y[:,1:].reshape(-1)
        
        loss = torch.nn.functional.cross_entropy(logits,target)
        return loss
    
    def generate(self, y):
        device = next(self.parameters()).device
        self.eval()        
        y = y.tolist()
        
        with torch.no_grad():                        
            while y[-1] != 22 and len(y) < self.seq_len:
                logits = self.decoder(torch.tensor(y).reshape(1,-1).to(device))
                y.append(logits.argmax(-1)[:,-1].item())
                
        return y

NameError: name 'torch' is not defined

test a small model and loss with artifical data

In [ ]:
model = Transformer(vocab_size=24, nb_layers=4, d_model=512, n_heads=2, d_head=32, dropout=0.1, seq_len=32)
x = torch.randint(0, 24, (1, 32))
print( model(x).shape )
print( model.loss(x) )

Causal Self Attention, d_model: 512 n_heads: 2 d_head: 32 seq_len: 32
Causal Self Attention, d_model: 512 n_heads: 2 d_head: 32 seq_len: 32
Causal Self Attention, d_model: 512 n_heads: 2 d_head: 32 seq_len: 32
Causal Self Attention, d_model: 512 n_heads: 2 d_head: 32 seq_len: 32
torch.Size([1, 32, 24])
tensor(3.3458, grad_fn=<NllLossBackward0>)


## Análisis del vocabulario de fechas2

Primero extraemos todas las palabras únicas del dataset para construir el tokenizador apropiado.

In [ ]:
# Extraer vocabulario único de los archivos de datos
vocab = set()

# Leer archivo de entrenamiento
with open('fechas2/fechas2_train.es.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()[1:]  # Saltar el encabezado
    for line in lines:
        parts = line.strip().split(',', 1)
        if len(parts) == 2:
            text = parts[1]
            vocab.update(text.split())

# Leer archivo de test
with open('fechas2/fechas2_test.es.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()[1:]  # Saltar el encabezado
    for line in lines:
        parts = line.strip().split(',', 1)
        if len(parts) == 2:
            text = parts[1]
            vocab.update(text.split())

# Ordenar vocabulario
vocab_sorted = sorted(vocab)

print(f"Total de palabras únicas: {len(vocab_sorted)}")
print(f"\nVocabulario completo:")
for i, word in enumerate(vocab_sorted):
    print(f"  {i}: '{word}'")

### Análisis de secuencias

Verificar longitud máxima de las secuencias para configurar seq_len apropiadamente.

In [ ]:
# Analizar longitud de secuencias
max_len = 0
min_len = float('inf')
lengths = []

for file in ['fechas2/fechas2_train.es.csv', 'fechas2/fechas2_test.es.csv']:
    with open(file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[1:]
        for line in lines:
            parts = line.strip().split(',', 1)
            if len(parts) == 2:
                words = parts[1].split()
                seq_len = len(words)
                lengths.append(seq_len)
                max_len = max(max_len, seq_len)
                min_len = min(min_len, seq_len)

print(f"Longitud mínima de secuencia: {min_len} palabras")
print(f"Longitud máxima de secuencia: {max_len} palabras")
print(f"Longitud promedio: {sum(lengths)/len(lengths):.2f} palabras")
print(f"\nCon tokens especiales (<sos>, <eos>) la longitud máxima será: {max_len + 2}")
print(f"Recomendación: usar seq_len={max_len + 4} (deja margen para padding)")

### Muestras de los datos

Veamos algunos ejemplos del dataset para entender su formato.

In [ ]:
# Mostrar ejemplos del dataset
print("=== EJEMPLOS DEL DATASET DE ENTRENAMIENTO ===\n")
with open('fechas2/fechas2_train.es.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()[1:11]  # Primeras 10 líneas
    for i, line in enumerate(lines, 1):
        parts = line.strip().split(',', 1)
        if len(parts) == 2:
            wav_path, text = parts
            print(f"{i}. Archivo: {wav_path}")
            print(f"   Texto: '{text}'")
            print(f"   Palabras: {text.split()}")
            print()

In [ ]:
class DigitSumTokenizer():    
    def __init__(self):
        self.word2index = {
            '0': 0,
            '1': 1,
            '2': 2,
            '3': 3,
            '4': 4,
            '5': 5,
            '6' :6,
            '7': 7,
            '8': 8,
            '9': 9,
            '+': 10,
            '=': 11,
            '<pad>': 20,
            '<sos>': 21,
            '<eos>': 22,    
        }       
        self.index2word = {v:k for k,v in self.word2index.items()}
    
    def encode(self, x, seq_len=-1):
        x = '<sos> ' + x + ' <eos>'
        x = [self.word2index[w] for w in x.split()]
        if seq_len > len(x):
            x = x + [self.word2index['<pad>']] * (seq_len - len(x))
        return torch.tensor(x)

    def decode(self, x):
        if isinstance(x, torch.Tensor):
            x = x.tolist()
        x = ' '.join([self.index2word[i] for i in x])
        x = x.replace('<sos>', '').replace('<eos>', '').replace('<pad>', '')
        return x.strip()
    
tokenizer = DigitSumTokenizer()

class Digitsumset(torch.utils.data.Dataset):
    def __init__(self, file, seq_len=16):
        super().__init__()
        self.seq_len = seq_len
        with open(file) as f:
            self.data = [line.strip().split(',') for line in f.readlines()]
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x, y = self.data[idx]
        return x, y, tokenizer.encode(y, seq_len=self.seq_len)

trainset = Digitsumset('fechas2/fechas2_train.es.csv')
testset = Digitsumset('fechas2/fechas2_test.es.csv')

test the access to a sample of the dataset

In [ ]:
x, y, yencoded = trainset[0]
print('input text:', x)
print('output text:', y)
print('output encoded:', yencoded, yencoded.shape)

input text: 6 + 4 + 9 + 9 =
output text: 6 + 4 + 9 + 9 = 2 8
output encoded: tensor([21,  6, 10,  4, 10,  9, 10,  9, 11,  2,  8, 22, 20, 20, 20, 20]) torch.Size([16])


# Train the network

In [ ]:
model = Transformer(vocab_size=24, 
                    nb_layers=4, 
                    d_model=128, d_ff=256, 
                    n_heads=8, d_head=64,
                    dropout=0.1, 
                    seq_len=16)

device = 'cuda'
model.to(device)
# opt = torch.optim.Adam(model.parameters(), lr=1e-3)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

nb_epochs = 15
batch_size = 32
model.train()

trainloader = torch.utils.data.DataLoader(trainset, batch_size, shuffle=True)
for e in range(nb_epochs):
    avg_loss = 0
    for _,_,x in trainloader:
        x = x.to(device)
        opt.zero_grad()
        loss = model.loss(x)
        loss.backward()
        opt.step()
        avg_loss += loss.item()
    print('epoch %d/%d: avg_loss: %.2f' % (e,nb_epochs,avg_loss/len(trainloader)))
        
    
torch.save( [model, opt], 'model11.pt')

Causal Self Attention, d_model: 128 n_heads: 8 d_head: 64 seq_len: 16
Causal Self Attention, d_model: 128 n_heads: 8 d_head: 64 seq_len: 16
Causal Self Attention, d_model: 128 n_heads: 8 d_head: 64 seq_len: 16
Causal Self Attention, d_model: 128 n_heads: 8 d_head: 64 seq_len: 16
epoch 0/15: avg_loss: 0.89
epoch 1/15: avg_loss: 0.75
epoch 2/15: avg_loss: 0.71
epoch 3/15: avg_loss: 0.70
epoch 4/15: avg_loss: 0.69
epoch 5/15: avg_loss: 0.68
epoch 6/15: avg_loss: 0.68
epoch 7/15: avg_loss: 0.67
epoch 8/15: avg_loss: 0.67
epoch 9/15: avg_loss: 0.66
epoch 10/15: avg_loss: 0.66
epoch 11/15: avg_loss: 0.66
epoch 12/15: avg_loss: 0.66
epoch 13/15: avg_loss: 0.65
epoch 14/15: avg_loss: 0.65


# Test the network

In [ ]:
#[model, opt] = torch.load('model11.pt')

# some samples
y = model.generate( tokenizer.encode( '1 + 1 + 8 + 9 =' )[:-1])
print( y )
print( tokenizer.decode(y) )

y = model.generate( tokenizer.encode('9 + 1 + 2 + 9 =' )[:-1])
print( y )
print( tokenizer.decode(y) )

[21, 1, 10, 1, 10, 8, 10, 9, 11, 1, 9, 22]
1 + 1 + 8 + 9 = 1 9
[21, 9, 10, 1, 10, 2, 10, 9, 11, 2, 1, 22]
9 + 1 + 2 + 9 = 2 1


## Error rate in the test set

In [ ]:
model.eval()
err = 0
for i, (txt_in, txt_out, _) in enumerate(testset):   
    y_pred = model.generate(tokenizer.encode(txt_in)[:-1])
    
    if txt_out != tokenizer.decode(y_pred):
        err += 1
print(f'error rate {err/len(testset):.2%},  ({err}/{len(testset)})')

error rate 7.48%,  (83/1110)
